# Commutative Algebra Foundations v2

We develop the local commutative-algebra theory before transporting any construction through `Spec`. The primary objects are commutative rings, actual ring morphisms, quotient and localization universal arrows, local rings, fraction fields, $R$-algebras as a coslice category, free commutative algebras, and finite limits and colimits in $\mathbf{CRing}$.

In [18]:
from sage.all import *
from sage.categories.category import Category
from sage.categories.category_with_axiom import CategoryWithAxiom
from sage.categories.covariant_functorial_construction import (
    CovariantConstructionCategory,
    CovariantFunctorialConstruction,
)
from sage.categories.homset import Hom, Homset
from sage.categories.morphism import Morphism
from sage.categories.rings import Rings
from sage.categories.commutative_rings import CommutativeRings
from sage.categories.fields import Fields
from sage.categories.integral_domains import IntegralDomains
from sage.categories.algebras import Algebras
from sage.categories.commutative_algebras import CommutativeAlgebras
from sage.categories.map import FormalCompositeMap, Map
from sage.misc.cachefunc import cached_method
from sage.rings.morphism import RingHomomorphism
from sage.rings.polynomial.multi_polynomial_ring_base import MPolynomialRing_base
from sage.rings.polynomial.polynomial_ring import PolynomialRing_generic
from sage.structure.element import Element
from sage.structure.parent import Parent
from sage.structure.sage_object import SageObject

print('Sage version =', sage.version.version)
print('base category =', CommutativeRings())

Sage version = 10.10.beta0
base category = Category of commutative rings


## Arrows and commutative algebras over a base

The arrow category $\operatorname{Ar}(\mathbf{CRing})$ has ring morphisms as objects and commuting squares as morphisms. For a fixed commutative ring $R$, the coslice $(R\downarrow\mathbf{CRing})$ is the category of commutative $R$-algebras, including noncanonical structure morphisms.

In [3]:
def _native_equality_proves_equal(left, right):
    if left is right:
        return True
    try:
        return True if left == right else None
    except (TypeError, ValueError, AttributeError, NotImplementedError):
        return None


def _morphism_is_identity(morphism):
    if hasattr(morphism, 'is_identity'):
        try:
            return True if morphism.is_identity() else False
        except (TypeError, ValueError, AttributeError, NotImplementedError):
            return None
    return None


class MorphismEqualityCertificate(SageObject):
    def domain(self):
        raise NotImplementedError

    def verify(self, left, right):
        raise NotImplementedError


class PolynomialGeneratorEqualityCertificate(MorphismEqualityCertificate):
    def __init__(self, polynomial_ring):
        if not isinstance(
            polynomial_ring,
            (MPolynomialRing_base, PolynomialRing_generic),
        ):
            raise TypeError(
                'the generator certificate requires a polynomial-ring domain'
            )
        self._domain = polynomial_ring

    def domain(self):
        return self._domain

    def verify(self, left, right):
        if left.domain() is not self._domain:
            return False
        if right.domain() is not self._domain:
            return False
        if left.codomain() is not right.codomain():
            return False
        if not all(
            left(generator) == right(generator)
            for generator in self._domain.gens()
        ):
            return False
        coefficient_ring = self._domain.base_ring()
        if coefficient_ring in (QQ, ZZ):
            return True
        coefficient_inclusion = self._domain.coerce_map_from(
            coefficient_ring
        )
        if coefficient_inclusion is None:
            return None
        left_on_coefficients = left * coefficient_inclusion
        right_on_coefficients = right * coefficient_inclusion
        if _native_equality_proves_equal(
            left_on_coefficients,
            right_on_coefficients,
        ) is True:
            return True
        if isinstance(
            coefficient_ring,
            (MPolynomialRing_base, PolynomialRing_generic),
        ):
            return PolynomialGeneratorEqualityCertificate(
                coefficient_ring
            ).verify(
                left_on_coefficients,
                right_on_coefficients,
            )
        return None


class ArrowCategoryFunctor(CovariantFunctorialConstruction):
    _functor_name = 'arrow_object'
    _functor_category = 'ArrowCategory'


class ArrowCategoryConstruction(CovariantConstructionCategory):
    _functor_category = 'ArrowCategory'
    _base_category_class = (Category,)

    def __init__(self, category):
        self._base_category = category
        self._args = tuple()
        self._morphism_equality_certificates = {}
        Category.__init__(self)

    def _repr_object_names(self):
        return (
            'arrows in '
            f'{self.base_category()._repr_object_names()}'
        )

    def arrow_category(self):
        return self

    def arrow_object(self, arrow):
        return ArrowObject(self, arrow)

    def register_morphism_equality_certificate(self, certificate):
        self._morphism_equality_certificates[
            id(certificate.domain())
        ] = certificate
        return self

    def morphisms_equal(self, left, right):
        if _native_equality_proves_equal(left, right) is True:
            return True
        if left.domain() is not right.domain():
            return False
        if left.codomain() is not right.codomain():
            return False
        if (
            _morphism_is_identity(left) is True
            and _morphism_is_identity(right) is True
        ):
            return True
        certificate = self._morphism_equality_certificates.get(
            id(left.domain())
        )
        if certificate is None:
            return None
        return certificate.verify(left, right)

    def square_status(
        self,
        source_arrow,
        target_arrow,
        source_leg,
        target_leg,
    ):
        try:
            left_composite = target_leg * source_arrow
            right_composite = target_arrow * source_leg
        except (TypeError, ValueError, AttributeError, NotImplementedError):
            return None
        return self.morphisms_equal(
            left_composite,
            right_composite,
        )


@cached_method
def _arrow_category(self):
    return ArrowCategoryConstruction(self)


Category.ArrowCategory = _arrow_category


class CosliceCategoryFunctor(CovariantFunctorialConstruction):
    _functor_name = 'coslice_object'
    _functor_category = 'CosliceCategory'

    def __init__(self, base_object):
        self._base_object = base_object

    def base_object(self):
        return self._base_object


class CosliceCategoryConstruction(CovariantConstructionCategory):
    _functor_category = 'CosliceCategory'
    _base_category_class = (Category,)

    def __init__(self, category, base_object):
        if base_object not in category:
            raise TypeError(
                'the fixed object is not an object of the base category'
            )
        self._base_category = category
        self._args = (base_object,)
        self._base_object = base_object
        Category.__init__(self)

    def base_object(self):
        return self._base_object

    def arrow_category(self):
        return self.base_category().ArrowCategory()

    def extra_super_categories(self):
        return [self.arrow_category()]

    def _repr_object_names(self):
        return (
            f'objects under {self._base_object} in '
            f'{self.base_category()._repr_object_names()}'
        )

    def coslice_object(self, structure_morphism):
        if structure_morphism.domain() is not self._base_object:
            raise ValueError(
                'a coslice object must have the fixed source'
            )
        return CosliceObject(self, structure_morphism)

    def initial_object(self):
        identity = self._base_object.Hom(
            self._base_object
        ).identity()
        return self.coslice_object(identity)

    def register_morphism_equality_certificate(self, certificate):
        self.arrow_category().register_morphism_equality_certificate(
            certificate
        )
        return self


@cached_method
def _coslice_category(self, base_object):
    return CosliceCategoryConstruction(self, base_object)


Category.CosliceCategory = _coslice_category


class ArrowObject(SageObject):
    _arrow_kind = 'arrow'

    def __init__(self, category, arrow):
        if not hasattr(arrow, 'domain') or not hasattr(arrow, 'codomain'):
            raise TypeError(
                'an arrow object requires a categorical morphism'
            )
        if arrow.domain() not in category.base_category():
            raise TypeError(
                'the arrow source is not in the base category'
            )
        if arrow.codomain() not in category.base_category():
            raise TypeError(
                'the arrow target is not in the base category'
            )
        self._arrow = arrow
        self._construction_category = category
        self._presentations = {}

    def category(self):
        return self._construction_category

    def construction_category(self):
        return self._construction_category

    def arrow_category(self):
        return self._construction_category.arrow_category()

    def arrow(self):
        return self._arrow

    def source(self):
        return self._arrow.domain()

    def target(self):
        return self._arrow.codomain()

    def arrow_kind(self):
        return self._arrow_kind

    def structure_morphism(self):
        return self._arrow

    def underlying_object(self):
        if self._arrow_kind == 'coslice':
            return self.target()
        raise AttributeError(
            'a general arrow has no distinguished underlying endpoint'
        )

    def register_presentation(self, presentation):
        self._presentations[presentation.kind] = presentation
        return self

    def presentation(self, kind):
        return self._presentations.get(kind)

    def _repr_(self):
        return f'Arrow object ({self._arrow})'

    def _Hom_(self, codomain, category=None):
        if self._arrow_kind == 'coslice':
            return CosliceHomset(self, codomain)
        return ArrowHomset(self, codomain)


class CosliceObject(ArrowObject):
    _arrow_kind = 'coslice'


class ArrowHomset(Homset):
    def __init__(self, domain, codomain):
        if domain.construction_category() is not (
            codomain.construction_category()
        ):
            raise TypeError(
                'arrow morphisms require a common construction category'
            )
        Homset.__init__(
            self,
            domain,
            codomain,
            category=domain.construction_category(),
        )

    def _element_constructor_(self, legs):
        if not isinstance(legs, (tuple, list)) or len(legs) != 2:
            raise TypeError(
                'an arrow morphism requires source and target legs'
            )
        return self.from_legs(legs[0], legs[1])

    def from_legs(self, source_leg, target_leg):
        return ArrowMorphism(
            self,
            source_leg,
            target_leg,
        )

    def identity(self):
        if self.domain() is not self.codomain():
            raise TypeError(
                'the identity is defined only on an endomorphism homset'
            )
        source_identity = self.domain().source().Hom(
            self.domain().source()
        ).identity()
        target_identity = self.domain().target().Hom(
            self.domain().target()
        ).identity()
        return self.from_legs(
            source_identity,
            target_identity,
        )


class CosliceHomset(ArrowHomset):
    def _element_constructor_(self, target_leg):
        source_identity = self.domain().source().Hom(
            self.domain().source()
        ).identity()
        return self.from_legs(
            source_identity,
            target_leg,
        )

    def from_legs(self, source_leg, target_leg):
        identity = self.domain().source().Hom(
            self.domain().source()
        ).identity()
        status = _morphism_is_identity(source_leg)
        if status is None:
            status = self.domain().arrow_category().morphisms_equal(
                source_leg,
                identity,
            )
        if status is False:
            raise ValueError(
                'a coslice morphism must have identity source leg'
            )
        if status is None:
            raise NotImplementedError(
                'the source leg could not be certified as the identity'
            )
        return ArrowMorphism(
            self,
            source_leg,
            target_leg,
        )


class ArrowMorphism(Element):
    def __init__(
        self,
        parent,
        source_leg,
        target_leg,
        square_verified=False,
    ):
        source_arrow = parent.domain()
        target_arrow = parent.codomain()
        if source_leg.domain() is not source_arrow.source():
            raise ValueError(
                'the source leg has the wrong domain'
            )
        if source_leg.codomain() is not target_arrow.source():
            raise ValueError(
                'the source leg has the wrong codomain'
            )
        if target_leg.domain() is not source_arrow.target():
            raise ValueError(
                'the target leg has the wrong domain'
            )
        if target_leg.codomain() is not target_arrow.target():
            raise ValueError(
                'the target leg has the wrong codomain'
            )
        if not square_verified:
            status = source_arrow.arrow_category().square_status(
                source_arrow.arrow(),
                target_arrow.arrow(),
                source_leg,
                target_leg,
            )
            if status is False:
                raise ValueError(
                    'the arrow square does not commute'
                )
            if status is None:
                raise NotImplementedError(
                    'the arrow square could not be certified'
                )
        self._source_leg = source_leg
        self._target_leg = target_leg
        Element.__init__(self, parent)

    def domain(self):
        return self.parent().domain()

    def codomain(self):
        return self.parent().codomain()

    def source_leg(self):
        return self._source_leg

    def target_leg(self):
        return self._target_leg

    def underlying_morphism(self):
        if self.domain().arrow_kind() == 'coslice':
            return self._target_leg
        raise AttributeError(
            'a general arrow morphism has two primary legs'
        )

    def is_identity(self):
        if self.domain() is not self.codomain():
            return False
        arrow_category = self.domain().arrow_category()
        source_identity = self.domain().source().Hom(
            self.domain().source()
        ).identity()
        target_identity = self.domain().target().Hom(
            self.domain().target()
        ).identity()
        source_status = arrow_category.morphisms_equal(
            self._source_leg,
            source_identity,
        )
        target_status = arrow_category.morphisms_equal(
            self._target_leg,
            target_identity,
        )
        if source_status is False or target_status is False:
            return False
        if source_status is True and target_status is True:
            return True
        raise NotImplementedError(
            'identity of the arrow morphism could not be certified'
        )

    def __mul__(self, right):
        if not isinstance(right, ArrowMorphism):
            return NotImplemented
        if right.codomain() is not self.domain():
            raise TypeError(
                'the arrow morphisms are not composable'
            )
        try:
            if right.is_identity():
                return self
        except NotImplementedError:
            pass
        try:
            if self.is_identity():
                return right
        except NotImplementedError:
            pass
        return ArrowMorphism(
            Hom(right.domain(), self.codomain()),
            self._source_leg * right._source_leg,
            self._target_leg * right._target_leg,
            square_verified=True,
        )


print('Installed arrow and coslice categories for commutative algebra.')

Installed arrow and coslice categories for commutative algebra.


## Quotients, localizations, and fraction fields as universal arrows

A quotient is the morphism $R\to R/I$. A localization is the morphism $R\to S^{-1}R$ together with its factorization property. For an integral domain, the fraction field is the localization at the multiplicative subset of nonzero elements. The codomain ring parent is retained, but the universal arrow is the primary object.

In [ ]:
class MultiplicativeSubset(SageObject):
    def __init__(self, ring):
        if ring not in CommutativeRings():
            raise TypeError(
                'the ambient parent must be a commutative ring'
            )
        self._ring = ring

    def ring(self):
        return self._ring

    def contains(self, element):
        raise NotImplementedError


class FinitelyGeneratedMultiplicativeSubset(MultiplicativeSubset):
    def __init__(self, ring, generators):
        super().__init__(ring)
        self._generators = tuple(
            ring(generator)
            for generator in generators
        )
        if not self._generators:
            self._generators = (ring.one(),)
        if any(
            generator.is_zero()
            for generator in self._generators
        ):
            raise ValueError(
                'a multiplicative subset may not contain zero'
            )

    def generators(self):
        return self._generators

    def _repr_(self):
        return (
            f'Multiplicative subset of {self.ring()} '
            f'generated by {self._generators}'
        )


class ComplementOfPrimeIdeal(MultiplicativeSubset):
    def __init__(self, prime_ideal):
        ring = prime_ideal.ring()
        if not prime_ideal.is_prime():
            raise ValueError(
                'the ideal must be prime'
            )
        super().__init__(ring)
        self._prime_ideal = prime_ideal

    def prime_ideal(self):
        return self._prime_ideal

    def contains(self, element):
        return (
            self.ring()(element)
            not in self._prime_ideal
        )

    def _repr_(self):
        return (
            f'Complement of {self._prime_ideal} '
            f'in {self.ring()}'
        )


class NonzeroElements(MultiplicativeSubset):
    def __init__(self, domain):
        if domain not in IntegralDomains():
            raise TypeError(
                'the nonzero elements are multiplicative only in a domain'
            )
        super().__init__(domain)

    def contains(self, element):
        return not self.ring()(element).is_zero()

    def _repr_(self):
        return f'Nonzero elements of {self.ring()}'


class CanonicalLocalizationMorphism(RingHomomorphism):
    def __init__(self, ring, localization_ring):
        if localization_ring.base_ring() is not ring:
            raise ValueError(
                'the localization parent has the wrong base ring'
            )
        self._localization_ring = localization_ring
        RingHomomorphism.__init__(
            self,
            Hom(ring, localization_ring),
        )

    def _call_(self, element):
        return self._localization_ring(element)

    def _repr_defn(self):
        return 'Canonical localization morphism'


class LocalizationExtensionMorphism(RingHomomorphism):
    def __init__(
        self,
        localization_ring,
        target_ring,
        base_map,
        inverted_generators,
    ):
        if localization_ring.base_ring() is not base_map.domain():
            raise ValueError(
                'the base map has the wrong domain'
            )
        if target_ring is not base_map.codomain():
            raise ValueError(
                'the base map has the wrong codomain'
            )
        self._base_map = base_map
        self._inverted_generators = tuple(
            base_map.domain()(generator)
            for generator in inverted_generators
        )
        if not all(
            target_ring(base_map(generator)).is_unit()
            for generator in self._inverted_generators
        ):
            raise ValueError(
                'an inverted generator does not map to a unit'
            )
        RingHomomorphism.__init__(
            self,
            Hom(localization_ring, target_ring),
        )

    def base_map(self):
        return self._base_map

    def _call_(self, element):
        numerator_image = self._base_map(
            element.numerator()
        )
        denominator_image = self._base_map(
            element.denominator()
        )
        if not denominator_image.is_unit():
            raise ArithmeticError(
                'the denominator image is not a unit'
            )
        return (
            numerator_image
            * denominator_image.inverse_of_unit()
        )

    def _repr_defn(self):
        return (
            'Induced by the localization universal property'
        )


class LocalizationEqualityCertificate(MorphismEqualityCertificate):
    def __init__(self, localization_arrow, base_certificate):
        self._localization_arrow = localization_arrow
        self._domain = localization_arrow.codomain()
        self._base_certificate = base_certificate

    def domain(self):
        return self._domain

    def verify(self, left, right):
        if left.domain() is not self._domain:
            return False
        if right.domain() is not self._domain:
            return False
        if left.codomain() is not right.codomain():
            return False
        return self._base_certificate.verify(
            left * self._localization_arrow,
            right * self._localization_arrow,
        )


class LocalizationUniversalProperty(SageObject):
    def __init__(self, multiplicative_subset, localization_arrow):
        self._multiplicative_subset = multiplicative_subset
        self._arrow = localization_arrow

    def multiplicative_subset(self):
        return self._multiplicative_subset

    def arrow(self):
        return self._arrow

    def localization_ring(self):
        return self._arrow.codomain()

    def factor(self, base_map):
        if base_map.domain() is not self._arrow.domain():
            raise ValueError(
                'the map has the wrong source ring'
            )
        subset = self._multiplicative_subset
        if not isinstance(
            subset,
            FinitelyGeneratedMultiplicativeSubset,
        ):
            raise NotImplementedError(
                'the current factorization backend requires a finitely generated multiplicative subset'
            )
        return LocalizationExtensionMorphism(
            self.localization_ring(),
            base_map.codomain(),
            base_map,
            subset.generators(),
        )


class FractionFieldEmbeddingMorphism(RingHomomorphism):
    def __init__(self, domain, fraction_field):
        if domain not in IntegralDomains():
            raise TypeError(
                'a fraction field requires an integral domain'
            )
        self._fraction_field = fraction_field
        RingHomomorphism.__init__(
            self,
            Hom(domain, fraction_field),
        )

    def _call_(self, element):
        return self._fraction_field(element)

    def _repr_defn(self):
        return 'Canonical fraction-field embedding'


class FractionFieldExtensionMorphism(RingHomomorphism):
    def __init__(
        self,
        source_domain,
        fraction_field,
        target_field,
        base_map,
    ):
        if source_domain is not base_map.domain():
            raise ValueError(
                'the base map has the wrong domain'
            )
        if target_field not in Fields():
            raise TypeError(
                'the target must be a field'
            )
        if target_field is not base_map.codomain():
            raise ValueError(
                'the base map has the wrong codomain'
            )
        self._base_map = base_map
        RingHomomorphism.__init__(
            self,
            Hom(fraction_field, target_field),
        )

    def _call_(self, element):
        numerator_image = self._base_map(
            element.numerator()
        )
        denominator_image = self._base_map(
            element.denominator()
        )
        if denominator_image.is_zero():
            raise ZeroDivisionError(
                'the base map kills a denominator'
            )
        return numerator_image / denominator_image

    def _repr_defn(self):
        return (
            'Induced by the fraction-field universal property'
        )


class InjectiveMapToFieldCertificate(SageObject):
    def __init__(self, base_map, theorem=None):
        if base_map.domain() not in IntegralDomains():
            raise TypeError(
                'the source must be an integral domain'
            )
        if base_map.codomain() not in Fields():
            raise TypeError(
                'the target must be a field'
            )
        verified = isinstance(
            base_map,
            FractionFieldEmbeddingMorphism,
        )
        if not verified and hasattr(
            base_map,
            'is_injective',
        ):
            try:
                verified = (
                    base_map.is_injective() is True
                )
            except (
                NotImplementedError,
                TypeError,
                AttributeError,
            ):
                verified = False
        if not verified:
            raise NotImplementedError(
                'injectivity of the map into the field could not be certified'
            )
        self._base_map = base_map
        self._theorem = (
            str(theorem)
            if theorem is not None
            else 'injectivity verified by the morphism implementation'
        )

    def base_map(self):
        return self._base_map

    def theorem(self):
        return self._theorem


class FractionFieldUniversalProperty(LocalizationUniversalProperty):
    def factor(self, base_map, certificate):
        if certificate.base_map() is not base_map:
            raise ValueError(
                'the injectivity certificate belongs to another map'
            )
        return FractionFieldExtensionMorphism(
            self.arrow().domain(),
            self.localization_ring(),
            base_map.codomain(),
            base_map,
        )


class QuotientEqualityCertificate(MorphismEqualityCertificate):
    def __init__(self, quotient_arrow, base_certificate):
        self._quotient_arrow = quotient_arrow
        self._domain = quotient_arrow.codomain()
        self._base_certificate = base_certificate

    def domain(self):
        return self._domain

    def verify(self, left, right):
        if left.domain() is not self._domain:
            return False
        if right.domain() is not self._domain:
            return False
        if left.codomain() is not right.codomain():
            return False
        return self._base_certificate.verify(
            left * self._quotient_arrow,
            right * self._quotient_arrow,
        )


class QuotientUniversalProperty(SageObject):
    def __init__(self, ideal, quotient_arrow):
        if ideal.ring() is not quotient_arrow.domain():
            raise ValueError(
                'the ideal belongs to another ring'
            )
        self._ideal = ideal
        self._arrow = quotient_arrow

    def ideal(self):
        return self._ideal

    def arrow(self):
        return self._arrow

    def quotient_ring(self):
        return self._arrow.codomain()

    def factor(self, base_map):
        if base_map.domain() is not self._arrow.domain():
            raise ValueError(
                'the map has the wrong source ring'
            )
        if not all(
            base_map(generator).is_zero()
            for generator in self._ideal.gens()
        ):
            raise ValueError(
                'the map does not annihilate the quotient ideal'
            )
        quotient_ring = self.quotient_ring()
        source_ring = self._arrow.domain()
        if not hasattr(quotient_ring, 'gens'):
            raise NotImplementedError(
                'no finite-generator quotient factorization backend applies'
            )
        if len(quotient_ring.gens()) != len(source_ring.gens()):
            raise NotImplementedError(
                'the quotient generator presentation is incompatible with the source presentation'
            )
        generator_images = tuple(
            base_map(generator)
            for generator in source_ring.gens()
        )
        try:
            return quotient_ring.hom(
                generator_images,
                base_map.codomain(),
            )
        except Exception as error:
            raise NotImplementedError(
                'the native quotient homomorphism constructor rejected the finite presentation'
            ) from error


def _multiplicative_subset(self, generators):
    return FinitelyGeneratedMultiplicativeSubset(
        self,
        generators,
    )


def _localization_morphism(self, multiplicative_subset):
    if multiplicative_subset.ring() is not self:
        raise ValueError(
            'the multiplicative subset belongs to another ring'
        )
    if isinstance(
        multiplicative_subset,
        FinitelyGeneratedMultiplicativeSubset,
    ):
        localization_ring = self.localization(
            multiplicative_subset.generators()
        )
        arrow = CanonicalLocalizationMorphism(
            self,
            localization_ring,
        )
        return LocalizationUniversalProperty(
            multiplicative_subset,
            arrow,
        )
    if isinstance(
        multiplicative_subset,
        NonzeroElements,
    ):
        fraction_field = self.fraction_field()
        arrow = FractionFieldEmbeddingMorphism(
            self,
            fraction_field,
        )
        return FractionFieldUniversalProperty(
            multiplicative_subset,
            arrow,
        )
    if isinstance(
        multiplicative_subset,
        ComplementOfPrimeIdeal,
    ):
        raise NotImplementedError(
            'prime-complement localization is defined, but no faithful Sage parent backend is installed'
        )
    raise NotImplementedError(
        'no localization backend applies to this multiplicative subset'
    )


def _fraction_field_morphism(self):
    return self.localization_morphism(
        NonzeroElements(self)
    )


def _localization_at_prime(self, prime_ideal):
    return self.localization_morphism(
        ComplementOfPrimeIdeal(prime_ideal)
    )


def _quotient_morphism(self, ideal, names=None):
    if ideal.ring() is not self:
        raise ValueError(
            'the ideal belongs to another ring'
        )
    quotient_ring = (
        self.quotient(ideal)
        if names is None
        else self.quotient(
            ideal,
            names=names,
        )
    )
    quotient_arrow = quotient_ring.cover()
    if quotient_arrow.domain() is not self:
        raise NotImplementedError(
            'the native quotient parent does not expose the required source-ring quotient morphism'
        )
    return QuotientUniversalProperty(
        ideal,
        quotient_arrow,
    )


_CommutativeRingParentMethods = (
    CommutativeRings().parent_class
)
for method_name, method in (
    ('multiplicative_subset', _multiplicative_subset),
    ('localization_morphism', _localization_morphism),
    ('fraction_field_morphism', _fraction_field_morphism),
    ('localization_at_prime', _localization_at_prime),
    ('quotient_morphism', _quotient_morphism),
):
    if method_name in _CommutativeRingParentMethods.__dict__:
        raise RuntimeError(
            f'the commutative-ring category already defines {method_name}'
        )
    setattr(
        _CommutativeRingParentMethods,
        method_name,
        method,
    )

print('Installed quotient, localization, and fraction-field universal arrows.')

In [ ]:
R_local_regression = PolynomialRing(
    QQ,
    names=('x_local_regression', 'y_local_regression'),
)
x_local_regression, y_local_regression = (
    R_local_regression.gens()
)
base_certificate_local_regression = (
    PolynomialGeneratorEqualityCertificate(
        R_local_regression
    )
)

multiplicative_subset_regression = (
    R_local_regression.multiplicative_subset(
        (
            x_local_regression,
            y_local_regression,
        )
    )
)
localization_regression = (
    R_local_regression.localization_morphism(
        multiplicative_subset_regression
    )
)
ell_local_regression = (
    localization_regression.arrow()
)
L_local_regression = (
    localization_regression.localization_ring()
)
factor_local_regression = (
    localization_regression.factor(
        ell_local_regression
    )
)
localization_equality_regression = (
    LocalizationEqualityCertificate(
        ell_local_regression,
        base_certificate_local_regression,
    )
)
assert base_certificate_local_regression.verify(
    factor_local_regression
    * ell_local_regression,
    ell_local_regression,
)
assert localization_equality_regression.verify(
    factor_local_regression,
    L_local_regression.Hom(
        L_local_regression
    ).identity(),
)

R_quotient_regression = PolynomialRing(
    QQ,
    names=('z_quotient_regression',),
)
z_quotient_regression = (
    R_quotient_regression.gen()
)
I_quotient_regression = (
    R_quotient_regression.ideal(
        z_quotient_regression**2
    )
)
quotient_regression = (
    R_quotient_regression.quotient_morphism(
        I_quotient_regression,
        names=('zbar_quotient_regression',),
    )
)
q_quotient_regression = (
    quotient_regression.arrow()
)
evaluation_regression = (
    R_quotient_regression.hom(
        (QQ.zero(),),
        QQ,
    )
)
factor_quotient_regression = (
    quotient_regression.factor(
        evaluation_regression
    )
)
base_certificate_quotient_regression = (
    PolynomialGeneratorEqualityCertificate(
        R_quotient_regression
    )
)
quotient_equality_regression = (
    QuotientEqualityCertificate(
        q_quotient_regression,
        base_certificate_quotient_regression,
    )
)
assert base_certificate_quotient_regression.verify(
    factor_quotient_regression
    * q_quotient_regression,
    evaluation_regression,
)
assert quotient_equality_regression.verify(
    factor_quotient_regression,
    factor_quotient_regression,
)

R_fraction_regression = PolynomialRing(
    QQ,
    names=('t_fraction_regression',),
)
t_fraction_regression = (
    R_fraction_regression.gen()
)
fraction_regression = (
    R_fraction_regression.fraction_field_morphism()
)
ell_fraction_regression = (
    fraction_regression.arrow()
)
K_fraction_regression = (
    fraction_regression.localization_ring()
)
injective_fraction_regression = (
    InjectiveMapToFieldCertificate(
        ell_fraction_regression,
        theorem=(
            'The canonical morphism from an integral domain '
            'to its fraction field is injective.'
        ),
    )
)
factor_fraction_regression = (
    fraction_regression.factor(
        ell_fraction_regression,
        injective_fraction_regression,
    )
)
base_certificate_fraction_regression = (
    PolynomialGeneratorEqualityCertificate(
        R_fraction_regression
    )
)
fraction_equality_regression = (
    LocalizationEqualityCertificate(
        ell_fraction_regression,
        base_certificate_fraction_regression,
    )
)
assert base_certificate_fraction_regression.verify(
    factor_fraction_regression
    * ell_fraction_regression,
    ell_fraction_regression,
)
assert fraction_equality_regression.verify(
    factor_fraction_regression,
    K_fraction_regression.Hom(
        K_fraction_regression
    ).identity(),
)

prime_localization_blocked_regression = False
prime_ideal_regression = (
    R_fraction_regression.ideal(
        t_fraction_regression
    )
)
try:
    R_fraction_regression.localization_at_prime(
        prime_ideal_regression
    )
except NotImplementedError:
    prime_localization_blocked_regression = True
assert prime_localization_blocked_regression

print('Local universal-arrow regressions passed.')
print('localization ring =', L_local_regression)
print('quotient ring =', quotient_regression.quotient_ring())
print('fraction field =', K_fraction_regression)
print(
    'prime-complement localization explicitly gated =',
    prime_localization_blocked_regression,
)